# Chunking structurel (container_ast_chunker) + retriever pondéré — modèle de base uniquement

3e notebook, **séparé** de `colab_50_trials.ipynb` et `colab_weighted_ast_retrieval.ipynb` — autonome, ne dépend d'aucun des deux.

**Question posée** : le retriever pondéré + attention AST est déjà le plus prometteur mesuré jusqu'ici (39.0% EM à n=200 sur `codegen-2B`, contre 29.5-37.0% pour les autres signaux). Est-ce que le faire tourner sur des chunks **structurels** (fonctions/classes entières, jamais coupées au milieu d'une instruction, en-tête conservé) au lieu des fenêtres de lignes fixes de `ast_chunker` l'améliore encore ?

**3 conditions, modèle de base (`codegen-2B-mono`) uniquement** (pas d'instruct ici) :
1. Sans retrieval
2. Pondéré + attention AST sur `ast_chunker` (fenêtres glissantes — la référence déjà établie)
3. Pondéré + attention AST sur `container_ast_chunker` (chunks structurels — le nouveau test)

**Important** : si une cellule plante avec une erreur CUDA ("device-side assert"), redémarre la session avant de relancer.

## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Installer les dépendances

In [ ]:
!pip install -q scikit-learn editdistance transformers accelerate tree-sitter tree-sitter-python

## 1bis. (Optionnel) Token Hugging Face

Pas nécessaire pour `codegen-2B-mono` (modèle public) — évite juste le throttling anonyme.

In [ ]:
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé.")
except Exception:
    print("Pas de secret HF_TOKEN configuré — pas grave, pas nécessaire pour ce modèle public.")

## 2. Réécrire le code du projet (copié tel quel depuis `repocoder-mine/`)

In [ ]:
%%writefile dataset.py
import json
import os
import random
import re
from pathlib import Path
from typing import Any, Literal


REQUIRED_FIELDS = {"prompt", "groundtruth", "right_context"}
Split = Literal[
    "baseline",
    "bm25",
    "unixcoder",
    "openai",
    "oracle_bm25",
    "oracle_unixcoder",
    "oracle_openai",
]


def load_jsonl(file_path: str | Path) -> list[dict[str, Any]]:
    """Charger les exemples valides d'un fichier JSONL."""
    records = []
    path = Path(file_path)

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                print(f"Ligne ignorée {line_number}: JSON invalide ({error})")
                continue

            if not isinstance(record, dict):
                print(f"Ligne ignorée {line_number}: l'objet n'est pas un dictionnaire")
                continue

            missing_fields = REQUIRED_FIELDS - record.keys()
            if missing_fields:
                print(
                    f"Ligne ignorée {line_number}: champs manquants "
                    f"{sorted(missing_fields)}"
                )
                continue

            if any(not isinstance(record[field], str) for field in REQUIRED_FIELDS):
                print(f"Ligne ignorée {line_number}: un champ contient une valeur invalide")
                continue

            records.append(record)

    return records


def load_cceval_examples(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    sample: int | None = None,
    seed: int = 42,
):
    """Charger les exemples avec le modèle officiel de CrossCodeEval."""
    from cceval.dataset import load_cceval_dataset as load_official_dataset

    return load_official_dataset(
        path=str(path) if path is not None else None,
        language=language,
        split=split,
        sample=sample,
        seed=seed,
    )


def load_cceval_dataset(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    sample: int | None = None,
    seed: int = 42,
) -> list[dict[str, Any]]:
    """Charger le dataset CrossCodeEval avec ses champs d'évaluation."""
    if path is None:
        path = resolve_cceval_path(split, language)

    records = load_jsonl(path)
    task_ids = [record.get("metadata", {}).get("task_id") for record in records]
    task_ids = [task_id for task_id in task_ids if task_id is not None]
    if len(task_ids) != len(set(task_ids)):
        raise ValueError("Le dataset contient des task_id en double")

    if sample is not None:
        if sample < 0:
            raise ValueError("sample doit être positif")
        records = random.Random(seed).sample(records, min(sample, len(records)))

    return records


def resolve_cceval_path(split: Split = "baseline", language: str = "python") -> Path:
    """Construire le chemin CrossCodeEval depuis CCEVAL_DATA_DIR."""
    data_dir = os.environ.get("CCEVAL_DATA_DIR")
    if data_dir is None:
        raise ValueError("La variable CCEVAL_DATA_DIR n'est pas définie")

    filenames = {
        "baseline": "line_completion.jsonl",
        "bm25": "line_completion_rg1_bm25.jsonl",
        "unixcoder": "line_completion_rg1_unixcoder_cosine_sim.jsonl",
        "openai": "line_completion_rg1_openai_cosine_sim.jsonl",
        "oracle_bm25": "line_completion_oracle_bm25.jsonl",
        "oracle_unixcoder": "line_completion_oracle_unixcoder_cosine_sim.jsonl",
        "oracle_openai": "line_completion_oracle_openai_cosine_sim.jsonl",
    }
    return Path(data_dir) / language / filenames[split]


SLIDING_WINDOW_SIZE = 20  # S_w dans l'article RepoCoder
SLIDING_STRIDE = 10  # S_s dans l'article RepoCoder


def slide_over_text(
    text: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[str]:
    """Découper un texte en fenêtres glissantes de lignes (S_w, S_s de RepoCoder)."""
    if window_size <= 0:
        raise ValueError("window_size doit être supérieur à 0")
    if stride <= 0 or stride > window_size:
        raise ValueError("stride doit être compris entre 1 et window_size")

    lines = text.splitlines(keepends=True)
    if not lines:
        return []

    windows = []
    for start in range(0, len(lines), stride):
        window_text = "".join(lines[start:start + window_size]).strip()
        if window_text:
            windows.append(window_text)
        if start + window_size >= len(lines):
            break
    return windows


def last_lines(text: str, n: int = SLIDING_STRIDE) -> str:
    """Garder les n dernières lignes d'un texte (utilisé comme requête de retrieval)."""
    lines = text.splitlines(keepends=True)
    return "".join(lines[-n:])


def first_lines(text: str, n: int = SLIDING_STRIDE) -> str:
    """Garder les n premières lignes d'un texte (utilisé sur la prédiction précédente)."""
    lines = text.splitlines(keepends=True)
    return "".join(lines[:n])


def extract_repository_snippets(
    records: list[dict[str, Any]],
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> dict[str, list[dict[str, Any]]]:
    """Regrouper les exemples CCEval par dépôt puis extraire leurs fenêtres glissantes.

    Reproduit le découpage de RepoCoder (S_w=20, S_s=10 par défaut) : chaque
    dépôt (metadata.repository) reçoit la liste des morceaux de code obtenus
    en faisant glisser une fenêtre sur les lignes de chaque exemple.
    """
    repositories: dict[str, list[dict[str, Any]]] = {}

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository") if isinstance(metadata, dict) else None
        if not repository:
            continue

        file_content = record["prompt"] + record.get("right_context", "")
        snippets = slide_over_text(file_content, window_size=window_size, stride=stride)

        for snippet in snippets:
            repositories.setdefault(repository, []).append(
                {
                    "task_id": metadata.get("task_id"),
                    "file": metadata.get("file"),
                    "snippet": snippet,
                }
            )

    return repositories


def extract_cceval_repository_snippets(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
    sample: int | None = None,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Charger le dataset CCEval puis extraire les fenêtres glissantes par dépôt.

    Utilise par défaut les hyperparamètres de l'article RepoCoder
    (S_w=20, S_s=10) pour construire la base de code de chaque dépôt.
    """
    records = load_cceval_dataset(
        path=path, language=language, split=split, sample=sample, seed=seed
    )
    return extract_repository_snippets(records, window_size=window_size, stride=stride)


def save_repository_snippets(
    repositories: dict[str, list[dict[str, Any]]], output_dir: str | Path
) -> dict[str, int]:
    """Sauvegarder les fenêtres glissantes de chaque dépôt dans son propre fichier JSONL."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    counts = {}
    for repository, snippets in repositories.items():
        safe_name = re.sub(r"[^\w.-]", "_", repository)
        save_jsonl(snippets, output_path / f"{safe_name}.jsonl")
        counts[repository] = len(snippets)

    return counts


def prepare_repository_snippets(
    output_dir: str | Path,
    path: str | Path | None = None,
    language: str = "python",
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> dict[str, int]:
    """Extraire les fenêtres glissantes du split baseline (line_completion.jsonl) et les sauvegarder par dépôt."""
    repositories = extract_cceval_repository_snippets(
        path=path,
        language=language,
        split="baseline",
        window_size=window_size,
        stride=stride,
    )
    return save_repository_snippets(repositories, output_dir)


def split_dataset(
    records: list[dict[str, Any]],
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Mélanger les exemples puis créer les ensembles train/validation/test."""
    if train_ratio <= 0 or validation_ratio < 0:
        raise ValueError("Les ratios doivent être positifs")
    if train_ratio + validation_ratio >= 1:
        raise ValueError("La somme des ratios doit être inférieure à 1")

    shuffled_records = records.copy()
    random.Random(seed).shuffle(shuffled_records)

    train_end = int(len(shuffled_records) * train_ratio)
    validation_end = train_end + int(len(shuffled_records) * validation_ratio)

    return {
        "train": shuffled_records[:train_end],
        "validation": shuffled_records[train_end:validation_end],
        "test": shuffled_records[validation_end:],
    }


def split_by_repository(
    records: list[dict[str, Any]],
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Séparer les dépôts entiers pour éviter une fuite entre les ensembles."""
    repositories = {}
    records_without_repository = []

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository") if isinstance(metadata, dict) else None
        if repository:
            repositories.setdefault(repository, []).append(record)
        else:
            records_without_repository.append(record)

    if not repositories:
        return split_dataset(records, train_ratio, validation_ratio, seed)

    repository_names = list(repositories)
    random.Random(seed).shuffle(repository_names)
    train_repository_end = max(1, int(len(repository_names) * train_ratio))
    validation_repository_end = train_repository_end + int(
        len(repository_names) * validation_ratio
    )

    splits = {
        "train": [],
        "validation": [],
        "test": [],
    }
    for repository in repository_names[:train_repository_end]:
        splits["train"].extend(repositories[repository])
    for repository in repository_names[train_repository_end:validation_repository_end]:
        splits["validation"].extend(repositories[repository])
    for repository in repository_names[validation_repository_end:]:
        splits["test"].extend(repositories[repository])

    # Les exemples sans dépôt sont répartis uniquement après le découpage principal.
    splits["train"].extend(records_without_repository)
    return splits


def save_jsonl(records: list[dict[str, Any]], file_path: str | Path) -> None:
    """Sauvegarder une liste d'exemples au format JSONL."""
    path = Path(file_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


def prepare_dataset(input_path: str | Path, output_dir: str | Path) -> dict[str, int]:
    """Charger, séparer et sauvegarder le dataset dans trois fichiers."""
    records = load_jsonl(input_path)
    splits = split_by_repository(records)
    output_path = Path(output_dir)

    for split_name, split_records in splits.items():
        save_jsonl(split_records, output_path / f"{split_name}.jsonl")

    return {split_name: len(split_records) for split_name, split_records in splits.items()}


if __name__ == "__main__":
    project_dir = Path(__file__).resolve().parent
    source = Path(r"C:\Users\User\Downloads\line_completion.jsonl")
    destination = project_dir / "data"
    counts = prepare_dataset(source, destination)

    print("Dataset organisé :")
    for split_name, count in counts.items():
        print(f"- {split_name}: {count} exemples")

    # Première extraction : fenêtres glissantes (S_w=20, S_s=10) par dépôt,
    # à partir de line_completion.jsonl uniquement, sauvegardées dans un dossier dédié.
    repository_destination = project_dir / "data" / "repositories"
    repository_counts = prepare_repository_snippets(repository_destination, path=source)

    print(f"\nFenêtres glissantes sauvegardées dans {repository_destination} :")
    for repository, count in repository_counts.items():
        print(f"- {repository}: {count} fenêtres")

In [ ]:
%%writefile ast_chunker.py
import ast
import hashlib
import os
import pickle
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dataset import SLIDING_STRIDE, SLIDING_WINDOW_SIZE

PROJECT_DIR = Path(__file__).resolve().parent
DEFAULT_CACHE_DIR = PROJECT_DIR / "data" / "cache" / "ast_chunks"


@dataclass
class ScopeBlock:
    """Un bloc nommé (classe ou fonction/méthode) avec ses bornes de lignes et ses identifiants propres."""

    kind: str  # "class" ou "function"
    name: str
    line_start: int
    line_end: int
    identifiers: set[str] = field(default_factory=set)


class _LocalNamesCollector(ast.NodeVisitor):
    """Collecte les noms assignés/définis dans un scope, sans descendre dans les classes/fonctions imbriquées
    (elles ont leur propre ScopeBlock)."""

    def __init__(self) -> None:
        self.names: set[str] = set()

    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:
        self.names.add(node.name)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:
        self.names.add(node.name)

    def visit_ClassDef(self, node: ast.ClassDef) -> None:
        self.names.add(node.name)

    def visit_arg(self, node: ast.arg) -> None:
        self.names.add(node.arg)

    def visit_Name(self, node: ast.Name) -> None:
        if isinstance(node.ctx, ast.Store):
            self.names.add(node.id)

    def visit_ExceptHandler(self, node: ast.ExceptHandler) -> None:
        if node.name:
            self.names.add(node.name)
        self.generic_visit(node)

    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            self.names.add(alias.asname or alias.name.split(".")[0])

    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        for alias in node.names:
            self.names.add(alias.asname or alias.name)


def _collect_local_names(node: ast.AST) -> set[str]:
    """Noms locaux à un bloc (paramètres, variables assignées, imports locaux, fonctions/classes imbriquées)."""
    collector = _LocalNamesCollector()
    collector.generic_visit(node)  # generic_visit: visite les enfants, pas node lui-même
    return collector.names


def _collect_class_attributes(class_node: ast.ClassDef) -> set[str]:
    """Attributs de classe (`x = 1` dans le corps) et d'instance (`self.x = ...` dans les méthodes)."""
    attributes: set[str] = set()

    for stmt in class_node.body:
        if isinstance(stmt, ast.Assign):
            for target in stmt.targets:
                if isinstance(target, ast.Name):
                    attributes.add(target.id)
        elif isinstance(stmt, ast.AnnAssign) and isinstance(stmt.target, ast.Name):
            attributes.add(stmt.target.id)

    for node in ast.walk(class_node):
        if isinstance(node, ast.Attribute) and isinstance(node.ctx, ast.Store) and isinstance(node.value, ast.Name):
            attributes.add(node.attr)

    return attributes


def build_scope_map(code: str) -> tuple[set[str], list[ScopeBlock]]:
    """Analyser le code d'un fichier et retourner (imports du module, blocs classes/fonctions).

    Les imports de haut niveau (pas dans une fonction/classe) sont visibles dans
    tout le fichier. Chaque classe et chaque fonction/méthode devient un
    ScopeBlock avec ses propres identifiants (nom, attributs/paramètres,
    variables locales) et ses bornes de lignes (`node.lineno`/`node.end_lineno`).
    """
    tree = ast.parse(code)
    module_imports: set[str] = set()
    blocks: list[ScopeBlock] = []

    def visit(node: ast.AST, inside_def: bool) -> None:
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.Import, ast.ImportFrom)) and not inside_def:
                if isinstance(child, ast.Import):
                    for alias in child.names:
                        module_imports.add(alias.asname or alias.name.split(".")[0])
                else:
                    for alias in child.names:
                        module_imports.add(alias.asname or alias.name)

            if isinstance(child, ast.ClassDef):
                own_methods = {
                    n.name for n in child.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
                }
                identifiers = {child.name} | _collect_class_attributes(child) | own_methods
                blocks.append(ScopeBlock("class", child.name, child.lineno, child.end_lineno, identifiers))
                visit(child, True)
            elif isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                identifiers = {child.name} | _collect_local_names(child)
                blocks.append(ScopeBlock("function", child.name, child.lineno, child.end_lineno, identifiers))
                visit(child, True)
            else:
                visit(child, inside_def)

    visit(tree, False)
    return module_imports, blocks


def chunk_file_ast(
    file_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[dict[str, Any]]:
    """Découper un fichier Python en fenêtres glissantes de lignes enrichies par l'AST.

    Chaque fenêtre hérite des identifiants de TOUS les blocs (classe et/ou
    fonction(s)) dont l'intervalle de lignes chevauche la fenêtre, plus les
    imports du module. Une fenêtre à cheval sur deux méthodes hérite ainsi
    des identifiants des deux, afin qu'une requête touchant l'une ou l'autre
    puisse retrouver ce morceau.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            code = file.read()
    except OSError as error:
        print(f"Erreur de lecture de {file_path}: {error}")
        return []

    try:
        module_imports, blocks = build_scope_map(code)
    except SyntaxError as error:
        print(f"Fichier ignoré (syntaxe invalide) {file_path}: {error}")
        return []

    lines = code.splitlines()
    if not lines:
        return []

    chunks = []
    for start in range(0, len(lines), stride):
        end = min(start + window_size, len(lines))
        raw_code = "\n".join(lines[start:end]).strip()

        if raw_code:
            line_start, line_end = start + 1, end  # lignes 1-indexées, comme node.lineno

            identifiers = set(module_imports)
            for block in blocks:
                if max(line_start, block.line_start) <= min(line_end, block.line_end):
                    identifiers |= block.identifiers

            chunks.append(
                {
                    "file_path": file_path,
                    "line_start": line_start,
                    "line_end": line_end,
                    "raw_code": raw_code,
                    "identifiers": sorted(identifiers),
                }
            )

        if end >= len(lines):
            break

    return chunks


def load_and_chunk_repo_ast(
    dir_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[dict[str, Any]]:
    """Parcourir tous les fichiers .py d'un dossier et produire les chunks enrichis par AST."""
    all_chunks: list[dict[str, Any]] = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                all_chunks.extend(chunk_file_ast(file_path, window_size=window_size, stride=stride))
    return all_chunks


def _repo_fingerprint(dir_path: str) -> str:
    """Empreinte du contenu d'un dossier (chemin + date de modif + taille de chaque .py).

    Sert à invalider le cache automatiquement si un fichier source a changé,
    sans avoir à relire/hacher le contenu de chaque fichier.
    """
    entries = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                stat = os.stat(file_path)
                entries.append((os.path.relpath(file_path, dir_path), stat.st_mtime_ns, stat.st_size))
    entries.sort()
    return hashlib.sha1(repr(entries).encode("utf-8")).hexdigest()


def _cache_path(dir_path: str, window_size: int, stride: int, cache_dir: str | Path) -> Path:
    safe_name = re.sub(r"[^\w.-]", "_", os.path.normpath(os.path.abspath(dir_path)))
    return Path(cache_dir) / f"{safe_name}_ws{window_size}_stride{stride}.pkl"


def load_and_chunk_repo_ast_cached(
    dir_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
    cache_dir: str | Path = DEFAULT_CACHE_DIR,
) -> list[dict[str, Any]]:
    """Comme `load_and_chunk_repo_ast`, mais met le résultat en cache sur disque.

    Le cache est invalidé automatiquement si un fichier .py du dossier a été
    ajouté/modifié/supprimé depuis la dernière exécution (voir `_repo_fingerprint`),
    ou si `window_size`/`stride` changent (chaque combinaison a son propre fichier
    de cache).
    """
    cache_file = _cache_path(dir_path, window_size, stride, cache_dir)
    fingerprint = _repo_fingerprint(dir_path)

    if cache_file.exists():
        with cache_file.open("rb") as file:
            cached = pickle.load(file)
        if cached.get("fingerprint") == fingerprint:
            return cached["chunks"]

    chunks = load_and_chunk_repo_ast(dir_path, window_size=window_size, stride=stride)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    with cache_file.open("wb") as file:
        pickle.dump({"fingerprint": fingerprint, "chunks": chunks}, file)
    return chunks


if __name__ == "__main__":
    project_dir = os.path.dirname(os.path.abspath(__file__))
    demo_file = os.path.join(project_dir, "retriever.py")
    chunks = chunk_file_ast(demo_file)

    print(f"{len(chunks)} morceaux générés depuis {demo_file}\n")
    for chunk in chunks[:5]:
        print(f"--- lignes {chunk['line_start']}-{chunk['line_end']} ---")
        print("identifiants:", chunk["identifiers"])
        print()

In [ ]:
%%writefile ast_distance.py
"""Distance de sauts dans l'arbre AST, pour calculer l'attention statique
(attention_score = exp(-lambda * distance_ast)) attendue par
weighted_ast_scorer.weighted_ast_attention_score.

Module autonome : parse le VRAI fichier source complet (pas un extrait
tronqué, souvent syntaxiquement invalide isolément) et mesure la distance
entre l'occurrence la plus proche (avant le curseur) d'une variable et le
noeud englobant le curseur, via leur plus proche ancêtre commun (LCA).
"""

import ast
import math


def annotate_tree(root: ast.AST) -> tuple[dict[int, ast.AST], dict[int, int]]:
    """Parcourt l'arbre une fois et retourne (parent, depth), indexés par id(node).

    id(node) plutôt que node directement : les noeuds ast ne sont pas hashables
    de façon fiable pour tous les types, id() est stable et rapide.
    """
    parent: dict[int, ast.AST] = {}
    depth: dict[int, int] = {}
    stack: list[tuple[ast.AST, ast.AST | None, int]] = [(root, None, 0)]
    while stack:
        node, par, d = stack.pop()
        parent[id(node)] = par
        depth[id(node)] = d
        for child in ast.iter_child_nodes(node):
            stack.append((child, node, d + 1))
    return parent, depth


def find_cursor_node(root: ast.AST, depth: dict[int, int], line_no: int) -> ast.AST:
    """Le noeud le plus profond (le plus spécifique) dont l'intervalle de lignes
    contient line_no — représente \"où se trouve le curseur\" dans l'arbre."""
    best = root
    best_depth = -1
    for node in ast.walk(root):
        node_start = getattr(node, "lineno", None)
        node_end = getattr(node, "end_lineno", None)
        if node_start is not None and node_end is not None and node_start <= line_no <= node_end:
            d = depth[id(node)]
            if d > best_depth:
                best = node
                best_depth = d
    return best


def find_closest_occurrence(root: ast.AST, var_name: str, line_no: int) -> ast.AST | None:
    """La dernière occurrence (Name ou arg) de var_name strictement avant line_no.

    \"Dernière avant le curseur\" = la définition/usage le plus probablement
    pertinent pour compléter le code à cet endroit.
    """
    best_node = None
    best_line = -1
    for node in ast.walk(root):
        name = None
        if isinstance(node, ast.Name) and node.id == var_name:
            name = node.id
        elif isinstance(node, ast.arg) and node.arg == var_name:
            name = node.arg
        if name is None:
            continue
        node_line = getattr(node, "lineno", None)
        if node_line is not None and node_line <= line_no and node_line > best_line:
            best_line = node_line
            best_node = node
    return best_node


def tree_distance(
    node_a: ast.AST, node_b: ast.AST, parent: dict[int, ast.AST], depth: dict[int, int]
) -> int:
    """Nombre de sauts entre deux noeuds dans l'arbre, via leur plus proche
    ancêtre commun (LCA) : depth(a) + depth(b) - 2*depth(LCA)."""
    ancestors_a = set()
    node = node_a
    while node is not None:
        ancestors_a.add(id(node))
        node = parent.get(id(node))

    node = node_b
    steps_b = 0
    while node is not None and id(node) not in ancestors_a:
        node = parent.get(id(node))
        steps_b += 1

    if node is None:
        # pas d'ancêtre commun trouvé (ne devrait pas arriver dans un même arbre
        # connexe) -> distance maximale plutôt que planter
        return depth[id(node_a)] + depth[id(node_b)]

    lca_depth = depth[id(node)]
    return (depth[id(node_a)] - lca_depth) + steps_b


def compute_query_vars_with_attention(
    file_source: str,
    line_no: int,
    candidate_vars: set[str],
    lam: float = 0.1,
) -> dict[str, float]:
    """Pour chaque variable candidate présente avant line_no dans le vrai fichier,
    calcule attention_score = exp(-lambda * distance_ast) où distance_ast est la
    distance de sauts (LCA) entre sa dernière occurrence et le noeud du curseur.

    Une variable candidate absente du fichier (ou jamais utilisée avant line_no)
    est simplement omise du résultat (pas de score par défaut).
    """
    root = ast.parse(file_source)
    parent, depth = annotate_tree(root)
    cursor_node = find_cursor_node(root, depth, line_no)

    query_vars: dict[str, float] = {}
    for var_name in candidate_vars:
        occurrence = find_closest_occurrence(root, var_name, line_no)
        if occurrence is None:
            continue
        distance = tree_distance(occurrence, cursor_node, parent, depth)
        query_vars[var_name] = math.exp(-lam * distance)

    return query_vars

In [ ]:
%%writefile weighted_ast_scorer.py
"""Scoring par Théorie des Ensembles Pondérés avec Attention Statique AST.

Module autonome, non branché au pipeline existant (retriever.py) — pour
tester la formule avant intégration éventuelle. Vrai Jaccard pondéré
(intersection / union) : IDF précalculé x score d'attention AST (déjà
fourni par l'appelant, pas recalculé ici) x poids fixe selon le type de
symbole (variable de portée ou import) côté requête ; IDF seul côté
symboles présents uniquement dans le chunk.
"""


def weighted_ast_attention_score(
    query_vars: dict[str, float],
    query_imports: set[str],
    chunk_symbols: set[str],
    chunk_imports: set[str],
    doc_weights: dict[str, float],
    var_weight: float = 2.0,
    import_weight: float = 2.5,
) -> float:
    """Jaccard pondéré entre une requête et un chunk candidat.

    query_vars : {nom_variable: attention_score}, attention_score déjà
        calculé en amont comme exp(-lambda * distance_ast) (proximité dans
        l'arbre AST par rapport au curseur) — cette fonction ne fait que le
        consommer, pas le recalculer.
    query_imports : imports actifs au niveau du curseur.
    chunk_symbols : symboles AST du chunk candidat (comparés à query_vars).
    chunk_imports : imports du chunk candidat (comparés à query_imports).
    doc_weights : poids IDF précalculés par symbole ; 1.0 si absent.

    Numérateur (intersection) : pour chaque symbole de la requête (variable
    ou import) aussi présent dans le chunk,
        poids = doc_weights.get(symbole, 1.0) * multiplicateur_de_type
    (le multiplicateur inclut le score d'attention pour les variables — les
    imports n'ont pas de notion de distance AST, donc pas d'attention_score,
    seulement leur propre multiplicateur `import_weight`).

    Dénominateur (union) : la somme ci-dessus pour TOUS les symboles de la
    requête (matchés ou non) + la somme des poids des symboles du chunk qui
    ne sont PAS dans la requête, où pour ceux-ci poids = doc_weights.get(v,
    1.0) tel quel (pas de multiplicateur de type, pas d'attention — lecture
    littérale de la spécification : seul le côté requête a un
    multiplicateur de type explicite).

    Toujours entre 0.0 et 1.0 (l'intersection est une somme partielle des
    termes déjà comptés côté requête dans l'union — jamais de terme compté
    en trop). Retourne 0.0 si requête et chunk sont tous les deux vides.
    """
    intersection_weight = 0.0
    union_weight = 0.0

    for var, attention_score in query_vars.items():
        weight = doc_weights.get(var, 1.0) * attention_score * var_weight
        union_weight += weight
        if var in chunk_symbols:
            intersection_weight += weight

    for imp in query_imports:
        weight = doc_weights.get(imp, 1.0) * import_weight
        union_weight += weight
        if imp in chunk_imports:
            intersection_weight += weight

    for symbol in chunk_symbols:
        if symbol not in query_vars:
            union_weight += doc_weights.get(symbol, 1.0)

    for imp in chunk_imports:
        if imp not in query_imports:
            union_weight += doc_weights.get(imp, 1.0)

    if union_weight == 0.0:
        return 0.0

    return intersection_weight / union_weight

In [ ]:
%%writefile container_ast_chunker.py
"""Découpage du code par conteneurs AST (fonction/classe), pas par fenêtres
de lignes fixes — via Tree-Sitter, module autonome, non branché à
ast_chunker.py (qui reste la version scope-aware par fenêtres glissantes).

Principe :
- Un noeud fonction/classe/définition décorée assez petit devient UN chunk
  entier (jamais coupé au milieu d'une instruction ou d'une signature).
- Un noeud trop grand est découpé par ses instructions de haut niveau
  internes (if/for/try/... — jamais au milieu d'une instruction), chaque
  sous-chunk conservant l'en-tête (signature + décorateurs) et un léger
  chevauchement avec le sous-chunk précédent pour ne pas perdre les
  variables locales actives déclarées juste au-dessus.
- Le code au niveau module (imports, constantes, `if __name__ == ...`) qui
  n'est ni fonction ni classe est regroupé en chunks "module" séparés, pour
  ne rien perdre silencieusement.
"""

from tree_sitter import Language, Parser
import tree_sitter_python as tspython

PY_LANGUAGE = Language(tspython.language())
_parser = Parser(PY_LANGUAGE)

TARGET_NODES = {"function_definition", "class_definition", "decorated_definition"}


def _find_body_node(node):
    """Le noeud 'block' (corps) d'un function_definition/class_definition,
    ou None si le corps est sur la même ligne que la signature (rare)."""
    for child in node.children:
        if child.type == "block":
            return child
    return None


def _header_lines(node, lines):
    """Lignes de la signature (et des décorateurs pour une définition
    décorée), avant le début du corps."""
    if node.type == "decorated_definition":
        inner = node.children[-1]  # le dernier enfant est le function/class_definition
        decorator_end = inner.start_point[0] - 1
        decorator_lines = lines[node.start_point[0]:decorator_end + 1]
        return decorator_lines + _header_lines(inner, lines)

    body = _find_body_node(node)
    if body is None:
        return [lines[node.start_point[0]]]
    header_end = body.start_point[0] - 1
    return lines[node.start_point[0]:header_end + 1]


def _split_large_container(node, lines, max_lines, overlap_lines):
    """Découpe le CORPS d'un noeud trop grand en sous-chunks alignés sur ses
    instructions de haut niveau, avec en-tête conservé + chevauchement."""
    header = _header_lines(node, lines)
    header_text = "\n".join(header)

    body_owner = node.children[-1] if node.type == "decorated_definition" else node
    body = _find_body_node(body_owner)

    if body is None or not body.children:
        # corps sur une seule ligne ou vide -> rien à découper, un seul chunk tel quel
        start_line, end_line = node.start_point[0], node.end_point[0]
        return [{
            "start_line": start_line + 1,
            "end_line": end_line + 1,
            "type": node.type,
            "code": "\n".join(lines[start_line:end_line + 1]),
        }]

    statements = list(body.children)
    budget = max(1, max_lines - len(header))

    groups = []
    current = []
    for stmt in statements:
        if not current:
            current = [stmt]
            continue
        span = stmt.end_point[0] - current[0].start_point[0] + 1
        if span > budget:
            groups.append(current)
            current = [stmt]
        else:
            current.append(stmt)
    if current:
        groups.append(current)

    sub_chunks = []
    prev_end_line = None
    for group in groups:
        group_start = group[0].start_point[0]
        group_end = group[-1].end_point[0]

        overlap_text = []
        if prev_end_line is not None:
            overlap_start = max(0, prev_end_line - overlap_lines + 1)
            # ne pas empiéter sur le début du groupe courant lui-même
            overlap_end = min(prev_end_line, group_start - 1)
            if overlap_end >= overlap_start:
                overlap_text = lines[overlap_start:overlap_end + 1]

        body_text = lines[group_start:group_end + 1]
        code_text = "\n".join([header_text] + overlap_text + body_text)

        sub_chunks.append({
            "start_line": group_start + 1,
            "end_line": group_end + 1,
            "type": f"{node.type}:partial",
            "code": code_text,
        })
        prev_end_line = group_end

    return sub_chunks


def _pack_module_level_lines(nodes, lines, max_lines):
    """Regroupe des instructions consécutives de niveau module (ni fonction
    ni classe) en chunks de type 'module', bornés à max_lines."""
    if not nodes:
        return []
    chunks = []
    current = [nodes[0]]
    for node in nodes[1:]:
        span = node.end_point[0] - current[0].start_point[0] + 1
        if span > max_lines:
            chunks.append(current)
            current = [node]
        else:
            current.append(node)
    chunks.append(current)

    result = []
    for group in chunks:
        start_line, end_line = group[0].start_point[0], group[-1].end_point[0]
        result.append({
            "start_line": start_line + 1,
            "end_line": end_line + 1,
            "type": "module",
            "code": "\n".join(lines[start_line:end_line + 1]),
        })
    return result


def chunk_code_by_ast(code: str, max_lines: int = 30, overlap_lines: int = 3) -> list[dict]:
    """Découpe le code Python en respectant la structure AST (fonctions/classes).

    Ne coupe jamais au milieu d'une instruction ou d'une signature. Un noeud
    trop grand est découpé par ses instructions de haut niveau internes, en
    conservant l'en-tête et un chevauchement de contexte entre sous-chunks.
    Le code de niveau module (hors fonction/classe) est regroupé à part,
    jamais perdu silencieusement.
    """
    if not code.strip():
        return []

    tree = _parser.parse(bytes(code, "utf8"))
    root_node = tree.root_node
    lines = code.split("\n")
    chunks = []
    orphan_module_nodes = []

    def collect_chunks(node, is_root_level):
        if node.type in TARGET_NODES:
            start_line = node.start_point[0]
            end_line = node.end_point[0]
            chunk_len = end_line - start_line + 1

            if chunk_len <= max_lines:
                chunk_text = "\n".join(lines[start_line:end_line + 1])
                chunks.append({
                    "start_line": start_line + 1,
                    "end_line": end_line + 1,
                    "type": node.type,
                    "code": chunk_text,
                })
            else:
                chunks.extend(_split_large_container(node, lines, max_lines, overlap_lines))
            return

        if is_root_level and node is not root_node:
            orphan_module_nodes.append(node)
            return

        for child in node.children:
            collect_chunks(child, is_root_level=(node is root_node))

    collect_chunks(root_node, is_root_level=False)
    chunks.extend(_pack_module_level_lines(orphan_module_nodes, lines, max_lines))
    chunks.sort(key=lambda c: c["start_line"])

    return chunks

In [ ]:
%%writefile container_weighted_pipeline.py
"""Adapte container_ast_chunker (découpage par conteneurs AST : fonctions/
classes entières ou découpées par bloc interne) au format attendu par
weighted_ast_attention_score et par les fonctions communes du pipeline
(filter_safe_chunks, build_prompt, ...) : file_path, line_start, line_end,
raw_code, identifiers, chunk_imports.

Contrairement à ast_chunker (fenêtres de lignes fixes), chaque chunk produit
ici est un extrait de code généralement syntaxiquement valide seul (fonction/
méthode entière, ou en-tête + suite de statements complets) — on peut donc
réellement parser chaque chunk avec ast.parse pour en extraire ses propres
symboles/imports, sans avoir besoin de l'approximation "ensemble des imports
du dépôt entier" utilisée pour les chunks ast_chunker.
"""

import ast
import hashlib
import os
import pickle
import re
from pathlib import Path
from typing import Any

from container_ast_chunker import chunk_code_by_ast

PROJECT_DIR = Path(__file__).resolve().parent
DEFAULT_CACHE_DIR = PROJECT_DIR / "data" / "cache" / "container_chunks"

_IDENTIFIER_PATTERN = re.compile(r"[A-Za-z_]\w*")


def extract_chunk_identifiers(code_text: str) -> tuple[set[str], set[str]]:
    """(symboles, imports) d'un chunk. Tente un vrai parse AST (les chunks de
    container_ast_chunker sont généralement du Python valide isolément) ;
    retombe sur une extraction par regex si le parse échoue (ex. légère
    incohérence de recombinaison en-tête + chevauchement)."""
    try:
        tree = ast.parse(code_text)
    except SyntaxError:
        tokens = set(_IDENTIFIER_PATTERN.findall(code_text))
        return tokens, set()

    imports: set[str] = set()
    symbols: set[str] = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                imports.add(alias.asname or alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom):
            for alias in node.names:
                imports.add(alias.asname or alias.name)
        elif isinstance(node, ast.Name):
            symbols.add(node.id)
        elif isinstance(node, ast.arg):
            symbols.add(node.arg)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            symbols.add(node.name)

    symbols -= imports
    return symbols, imports


def chunk_file_container_ast(
    file_path: str, max_lines: int = 30, overlap_lines: int = 3
) -> list[dict[str, Any]]:
    try:
        code = open(file_path, "r", encoding="utf-8").read()
    except OSError as error:
        print(f"Erreur de lecture de {file_path}: {error}")
        return []

    try:
        raw_chunks = chunk_code_by_ast(code, max_lines=max_lines, overlap_lines=overlap_lines)
    except SyntaxError as error:
        print(f"Fichier ignoré (syntaxe invalide) {file_path}: {error}")
        return []

    chunks = []
    for raw_chunk in raw_chunks:
        symbols, imports = extract_chunk_identifiers(raw_chunk["code"])
        chunks.append({
            "file_path": file_path,
            "line_start": raw_chunk["start_line"],
            "line_end": raw_chunk["end_line"],
            "raw_code": raw_chunk["code"],
            "identifiers": sorted(symbols | imports),  # pour compute_doc_weights (IDF), comme ast_chunker
            "chunk_imports": sorted(imports),
        })
    return chunks


def load_and_chunk_repo_container_ast(
    dir_path: str, max_lines: int = 30, overlap_lines: int = 3
) -> list[dict[str, Any]]:
    all_chunks: list[dict[str, Any]] = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                all_chunks.extend(chunk_file_container_ast(file_path, max_lines=max_lines, overlap_lines=overlap_lines))
    return all_chunks


def _repo_fingerprint(dir_path: str) -> str:
    entries = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                stat = os.stat(file_path)
                entries.append((os.path.relpath(file_path, dir_path), stat.st_mtime_ns, stat.st_size))
    entries.sort()
    return hashlib.sha1(repr(entries).encode("utf-8")).hexdigest()


def load_and_chunk_repo_container_ast_cached(
    dir_path: str,
    max_lines: int = 30,
    overlap_lines: int = 3,
    cache_dir: str | Path = DEFAULT_CACHE_DIR,
) -> list[dict[str, Any]]:
    safe_name = re.sub(r"[^\w.-]", "_", os.path.normpath(os.path.abspath(dir_path)))
    cache_file = Path(cache_dir) / f"{safe_name}_max{max_lines}_ov{overlap_lines}.pkl"
    fingerprint = _repo_fingerprint(dir_path)

    if cache_file.exists():
        with cache_file.open("rb") as file:
            cached = pickle.load(file)
        if cached.get("fingerprint") == fingerprint:
            return cached["chunks"]

    chunks = load_and_chunk_repo_container_ast(dir_path, max_lines=max_lines, overlap_lines=overlap_lines)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    with cache_file.open("wb") as file:
        pickle.dump({"fingerprint": fingerprint, "chunks": chunks}, file)
    return chunks


def retrieve_top_k_weighted_container(
    query_vars: dict[str, float],
    query_imports: set[str],
    chunks: list[dict[str, Any]],
    doc_weights: dict[str, float],
    k: int = 10,
    var_weight: float = 2.0,
    import_weight: float = 2.5,
) -> list[dict[str, Any]]:
    """Comme retrieve_top_k_weighted, mais les imports du chunk viennent de
    son propre parse (chunk['chunk_imports']), pas d'une approximation par
    dépôt entier."""
    from weighted_ast_scorer import weighted_ast_attention_score

    scored = []
    for chunk in chunks:
        chunk_symbols = set(chunk["identifiers"]) - set(chunk["chunk_imports"])
        chunk_imports = set(chunk["chunk_imports"])
        score = weighted_ast_attention_score(
            query_vars, query_imports, chunk_symbols, chunk_imports, doc_weights,
            var_weight=var_weight, import_weight=import_weight,
        )
        scored.append({**chunk, "score": score})
    scored.sort(key=lambda item: item["score"], reverse=True)
    return scored[:k]

In [ ]:
%%writefile generator_min.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- copié tel quel depuis repocoder-mine/generator.py (call_huggingface_api) ---

_hf_model_cache: dict = {}


def call_huggingface_api(
    prompt: str,
    model: str = "Salesforce/codegen-2B-mono",
    max_new_tokens: int = 64,
    temperature: float = 0.0,
    max_prompt_tokens: int = 4096,
) -> str:
    """Générer une complétion avec un modèle Hugging Face chargé localement."""
    if model not in _hf_model_cache:
        tokenizer = AutoTokenizer.from_pretrained(model)
        tokenizer.truncation_side = "left"
        hf_model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        hf_model.eval()
        _hf_model_cache[model] = (tokenizer, hf_model)

    tokenizer, hf_model = _hf_model_cache[model]
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=max_prompt_tokens
    ).to(hf_model.device)

    with torch.no_grad():
        output_ids = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

## 3. Télécharger le dataset officiel RepoCoder + les 8 vrais dépôts

In [ ]:
!git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
%cd codet_src
!git sparse-checkout init --cone
!git sparse-checkout set RepoCoder
!git checkout main
%cd ..

In [ ]:
import zipfile

with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
    z.extractall('datasets_rapo')
with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
    z.extractall('repos_source')

print('Dataset et dépôts extraits.')

## 4. Charger les tâches (échantillon reproductible)

In [ ]:
import json
import random
from pathlib import Path

REPOS_DIR = Path('repos_source')
TASKS_PATH = Path('datasets_rapo/line_level_completion_1k_context_codegen.test.jsonl')
N_TRIALS = 50  # commence petit, augmente une fois que ça tourne
SEED = 42


def load_tasks(path):
    tasks = []
    with path.open('r', encoding='utf-8') as file:
        for line in file:
            if line.strip():
                tasks.append(json.loads(line))
    return tasks


all_tasks = load_tasks(TASKS_PATH)
sampled_tasks = random.Random(SEED).sample(all_tasks, N_TRIALS)
print(f'{len(all_tasks)} tâches disponibles, {len(sampled_tasks)} échantillonnées (seed={SEED}).')

## 5. Fonctions communes

Anti-fuite, troncature du code inachevé (`trim_code`), construction du prompt, scoring EM/ES — mêmes fonctions déjà validées dans les deux autres notebooks.

In [ ]:
import os
import editdistance


def filter_safe_chunks(chunks, repo_dir, fpath_tuple, context_start_lineno):
    task_file = os.path.normpath(str(repo_dir.joinpath(*fpath_tuple[1:])))
    safe = []
    for chunk in chunks:
        if os.path.normpath(chunk['file_path']) == task_file:
            if chunk['line_end'] - 1 > context_start_lineno:
                continue
        safe.append(chunk)
    return safe


def trim_code(text, max_chars=3000):
    if len(text) <= max_chars:
        return text
    trimmed = text[-max_chars:]
    newline_pos = trimmed.find('\n')
    return trimmed[newline_pos + 1:] if newline_pos != -1 else trimmed


def build_prompt(unfinished_code, retrieved_chunks, repo_dir, max_context_chars=3000):
    blocks = []
    total_chars = 0
    for chunk in retrieved_chunks:  # déjà trié du meilleur au moins bon
        rel_path = os.path.relpath(chunk['file_path'], repo_dir)
        block = f"# the below code fragment can be found in: {rel_path}\n{chunk['raw_code']}\n"
        if total_chars + len(block) > max_context_chars:
            break
        blocks.append(block)
        total_chars += len(block)
    context = '\n'.join(reversed(blocks))
    return f"{context}\n{unfinished_code}" if context else unfinished_code


def compute_em(target, prediction):
    target_lines = [line.strip() for line in target.splitlines() if line.strip()]
    prediction_lines = [line.strip() for line in prediction.splitlines() if line.strip()][:len(target_lines)]
    return int(target_lines == prediction_lines and len(target_lines) > 0)


def compute_es(target, prediction):
    target_lines = [line.strip() for line in target.splitlines() if line.strip()]
    target_str = '\n'.join(target_lines)
    prediction_lines = [line.strip() for line in prediction.splitlines() if line.strip()][:len(target_lines)]
    prediction_str = '\n'.join(prediction_lines)
    if not target_str and not prediction_str:
        return 1.0
    return 1 - (editdistance.eval(target_str, prediction_str) / max(len(target_str), len(prediction_str), 1))

## 6. Retriever pondéré sur les DEUX types de chunks

`weighted_sliding` : chunks `ast_chunker` (fenêtres glissantes) — imports séparés des symboles par approximation (ensemble des imports connus du dépôt entier), comme dans `colab_weighted_ast_retrieval.ipynb`.

`weighted_container` : chunks `container_ast_chunker` — chaque chunk est généralement du Python valide isolément, donc ses propres imports sont extraits par un vrai parse AST du chunk lui-même (plus précis, voir `container_weighted_pipeline.extract_chunk_identifiers`).

In [ ]:
import math
from collections import Counter

from ast_chunker import load_and_chunk_repo_ast_cached, build_scope_map
from ast_distance import compute_query_vars_with_attention
from weighted_ast_scorer import weighted_ast_attention_score
from container_weighted_pipeline import (
    load_and_chunk_repo_container_ast_cached,
    retrieve_top_k_weighted_container,
)
from generator_min import call_huggingface_api


def compute_doc_weights(chunks):
    n_docs = len(chunks)
    df = Counter()
    for chunk in chunks:
        df.update(set(chunk['identifiers']))
    return {symbol: math.log((n_docs + 1) / (count + 1)) + 1 for symbol, count in df.items()}


def compute_repo_import_names(repo_dir):
    import_names = set()
    for root, _, files in os.walk(repo_dir):
        for filename in files:
            if not filename.endswith('.py'):
                continue
            file_path = os.path.join(root, filename)
            try:
                code_text = open(file_path, 'r', encoding='utf-8').read()
                module_imports, _ = build_scope_map(code_text)
            except (SyntaxError, OSError):
                continue
            import_names |= module_imports
    return import_names


def split_chunk_symbols(chunk, repo_import_names):
    identifiers = set(chunk['identifiers'])
    chunk_imports = identifiers & repo_import_names
    chunk_symbols = identifiers - chunk_imports
    return chunk_symbols, chunk_imports


def retrieve_top_k_weighted_sliding(query_vars, query_imports, chunks, repo_import_names, doc_weights, k=10):
    scored = []
    for chunk in chunks:
        chunk_symbols, chunk_imports = split_chunk_symbols(chunk, repo_import_names)
        score = weighted_ast_attention_score(query_vars, query_imports, chunk_symbols, chunk_imports, doc_weights)
        scored.append({**chunk, 'score': score})
    scored.sort(key=lambda item: item['score'], reverse=True)
    return scored[:k]

## 7. Boucle générique : sans retrieval / pondéré-glissant / pondéré-structurel

In [ ]:
BASE_MODEL_NAME = 'Salesforce/codegen-2B-mono'
BASE_MAX_PROMPT_TOKENS = 2048 - 64 - 32  # n_positions=2048 pour codegen-*-mono

sliding_chunk_cache = {}
container_chunk_cache = {}
sliding_doc_weights_cache = {}
container_doc_weights_cache = {}
import_names_cache = {}


def run_condition(tasks, condition_name, retrieval_mode, k=10, max_code_chars=3000):
    condition_results = []
    for i, task in enumerate(tasks, start=1):
        metadata = task['metadata']
        repo = metadata['task_id'].split('/')[0]
        repo_dir = REPOS_DIR / repo
        unfinished_code = trim_code(task['prompt'], max_chars=max_code_chars)

        if retrieval_mode == 'none':
            retrieved = []
        else:
            target_file = repo_dir.joinpath(*metadata['fpath_tuple'][1:])
            file_source = target_file.read_text(encoding='utf-8')
            module_imports, blocks = build_scope_map(file_source)
            line_no = metadata['line_no']
            candidate_vars = set()
            for block in blocks:
                if block.line_start <= line_no <= block.line_end:
                    candidate_vars |= block.identifiers
            candidate_vars -= module_imports
            query_vars = compute_query_vars_with_attention(file_source, line_no, candidate_vars, lam=0.1)

            if retrieval_mode == 'weighted_sliding':
                if repo not in sliding_chunk_cache:
                    sliding_chunk_cache[repo] = load_and_chunk_repo_ast_cached(str(repo_dir))
                    sliding_doc_weights_cache[repo] = compute_doc_weights(sliding_chunk_cache[repo])
                    import_names_cache[repo] = compute_repo_import_names(repo_dir)
                safe_chunks = filter_safe_chunks(sliding_chunk_cache[repo], repo_dir, metadata['fpath_tuple'], metadata['context_start_lineno'])
                retrieved = retrieve_top_k_weighted_sliding(
                    query_vars, module_imports, safe_chunks, import_names_cache[repo], sliding_doc_weights_cache[repo], k=k
                )
            elif retrieval_mode == 'weighted_container':
                if repo not in container_chunk_cache:
                    container_chunk_cache[repo] = load_and_chunk_repo_container_ast_cached(str(repo_dir))
                    container_doc_weights_cache[repo] = compute_doc_weights(container_chunk_cache[repo])
                safe_chunks = filter_safe_chunks(container_chunk_cache[repo], repo_dir, metadata['fpath_tuple'], metadata['context_start_lineno'])
                retrieved = retrieve_top_k_weighted_container(
                    query_vars, module_imports, safe_chunks, container_doc_weights_cache[repo], k=k
                )
            else:
                raise ValueError(f'retrieval_mode inconnu: {retrieval_mode}')

        prompt = build_prompt(unfinished_code, retrieved, repo_dir) if retrieved else unfinished_code

        completion = call_huggingface_api(
            prompt, model=BASE_MODEL_NAME, max_new_tokens=64, max_prompt_tokens=BASE_MAX_PROMPT_TOKENS
        )
        em = compute_em(metadata['ground_truth'], completion)
        es = compute_es(metadata['ground_truth'], completion)
        condition_results.append({
            'task_id': metadata['task_id'],
            'ground_truth': metadata['ground_truth'],
            'completion': completion,
            'exact_match': em,
            'edit_similarity': es,
        })
        if i % 10 == 0 or i == len(tasks):
            print(f'[{condition_name}] {i}/{len(tasks)} — EM cumulé: '
                  f"{sum(r['exact_match'] for r in condition_results)}/{i}")
    return condition_results


def summarize(name, condition_results):
    em = sum(r['exact_match'] for r in condition_results)
    es = sum(r['edit_similarity'] for r in condition_results) / len(condition_results)
    n = len(condition_results)
    se = (em / n * (1 - em / n) / n) ** 0.5
    print(f"{name:<32}{em}/{n} ({100 * em / n:.1f}% ± {100 * 1.96 * se:.1f} pts, IC95%)"
          f"{'':>3}{es:>10.3f}")

## 8. Lancer les 3 conditions

In [ ]:
results = {}
results['none'] = run_condition(sampled_tasks, 'sans retrieval', 'none')
results['weighted_sliding'] = run_condition(sampled_tasks, 'pondéré (fenêtres glissantes)', 'weighted_sliding')
results['weighted_container'] = run_condition(sampled_tasks, 'pondéré (chunks structurels)', 'weighted_container')

## 9. Résultats

In [ ]:
for key, res in results.items():
    with open(f'results_{key}.jsonl', 'w', encoding='utf-8') as f:
        for r in res:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"{'':<32}{'Exact Match (IC95%)':>32}{'Edit Similarity':>18}")
summarize('Sans retrieval', results['none'])
summarize('Pondéré (fenêtres glissantes)', results['weighted_sliding'])
summarize('Pondéré (chunks structurels)', results['weighted_container'])